# Practice: Put an LLM in a Research Pipeline Safely

**Computational Text Analysis Pack · Module 27 — Generative Models in DH Research**

*Technique:* Designing, checking and reporting an LLM-assisted step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/14-generative-llms/27d_practice_llm_in_a_project.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **You want to use an LLM for one step of a real project — say, extracting
> which organisation each article is about. How do you do it so the result is
> checkable?**

The answer is the same discipline as any other instrument: define the task,
measure the error, report the protocol. What changes is that the instrument is
non-deterministic, which adds one requirement most people skip.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
LLM_BACKEND, _chat, _why = "recorded", None, ""
if not IN_COLAB:
    try:
        sys.path.insert(0, "/app")
        from utils.llm_gateway import gateway_chat as _gw
        def _chat(prompt, system=None, max_tokens=200):
            msgs = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
            txt, meta = _gw(msgs, model="gpt-oss-20b", max_tokens=max_tokens,
                            allow_fallback=False)
            return txt
        _chat("ok", max_tokens=5); LLM_BACKEND = "coursehub-gateway"
    except Exception as exc:
        _why = str(exc)[:70]
print(f"LLM backend: {LLM_BACKEND}" + (f"  ({_why})" if _why else ""))

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
# Step 1: define the task so a human could do it identically.
TASK = ("Extract the single organisation the article is mainly about. "
        "Reply with the organisation name only, or NONE if there is no clear one.")
SYSTEM = ("You are a careful research assistant extracting structured data. "
          "Answer with the requested value and nothing else. Never guess: if the "
          "text does not clearly support an answer, reply NONE.")

sample = df[(df["language"] == "English") & (df["n_words"] >= 60)].sample(
    n=12, random_state=5).reset_index(drop=True)
print(f"task defined; {len(sample)} documents to process\n")
print("TASK:", TASK)

In [ ]:
# Step 2: run it, and record the exact prompt with the output.
records = []
if LLM_BACKEND != "recorded":
    for _, r in sample.iterrows():
        body = " ".join(str(r["text"]).split())[:700]
        out = _chat(f"{TASK}\n\nArticle:\n{body}", system=SYSTEM, max_tokens=40)
        records.append({"headline": str(r["headline"])[:44],
                        "extracted": " ".join(out.split())[:40]})
    print(pd.DataFrame(records).to_string(index=False))
else:
    print("Pod/gateway offline — the checking protocol below is the lesson, and")
    print("it applies unchanged whichever model you use.")

In [ ]:
# Step 3: the requirement people skip — RUN IT TWICE.
if LLM_BACKEND != "recorded":
    second = []
    for _, r in sample.iterrows():
        body = " ".join(str(r["text"]).split())[:700]
        out = _chat(f"{TASK}\n\nArticle:\n{body}", system=SYSTEM, max_tokens=40)
        second.append(" ".join(out.split())[:40])
    agree = sum(1 for a, b in zip([x["extracted"] for x in records], second) if a == b)
    print(f"self-agreement across two identical runs: {agree}/{len(second)} "
          f"({agree/len(second):.0%})")
    print()
    print("This is the LLM's version of inter-annotator agreement, and it is a")
    print("CEILING on your accuracy: the model cannot be more right than it is")
    print("consistent. Report it exactly as module 17 reports kappa.")
    for a, b in zip([x["extracted"] for x in records], second):
        if a != b:
            print(f"   disagreed: {a!r} vs {b!r}")
else:
    print("Run twice, compare, and report the agreement rate. A model that gives")
    print("two different answers to the same prompt has an accuracy ceiling below")
    print("100% before you have measured anything against ground truth.")

In [ ]:
# Step 4: hand-check a sample against the text. There is no substitute.
print("HAND-CHECK PROTOCOL\n")
print("1. Take 20 outputs at random.")
print("2. For each, read the source and mark correct / wrong / unclear.")
print("3. Report precision, and the size of the sample you checked.")
print("4. Keep the marked sample; it is your evidence, and it is reusable when")
print("   you change model or prompt.")
print()
CHECKED, CORRECT = 20, 16
print(f"example: {CORRECT}/{CHECKED} correct -> precision {CORRECT/CHECKED:.2f}")
print()
plt.figure(figsize=(6.5, 3.2))
plt.barh(["self-agreement (ceiling)", "measured precision"], [0.85, CORRECT/CHECKED],
         color=["grey", "steelblue"])
plt.xlim(0, 1); plt.xlabel("proportion")
plt.title("You cannot be more accurate than you are consistent")
plt.tight_layout()

## What to report

**Defensible:** "Organisation names were extracted with [model, version, date]
using the prompt in Appendix A, temperature [t]. Self-agreement across two runs
on the same 12 items was [X]. A hand-checked random sample of 20 gave precision
[Y]. Extracted values are published with the dataset."

**Not defensible:** "we used an LLM to extract organisations." That sentence
contains no instrument, no error estimate, and no way to reproduce.

**Three rules for any LLM step in a pipeline:**

1. **Constrain the output.** A free-text answer cannot be checked automatically;
   a name, a label or NONE can.
2. **Run it twice and report agreement.** It is your accuracy ceiling.
3. **Never let it be the only reader of anything you will claim.** Module 27c
   showed the judge repeating the generator's error — the same applies when the
   model is both extractor and validator.

### 🔧 Your turn 1

Design an LLM step for your own project, then write the two sentences you would put in the methods section. Can a reader reproduce it?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The reproducibility test is brutal for LLM steps and worth applying honestly:
hosted models are deprecated and replaced, so a prompt that ran in one month may
be unreproducible the next. That is a real difference from every other method in
this course — a TF-IDF classifier from 2015 still runs. If your pipeline needs
to be reproducible in five years, an open model whose weights you can archive is
not a preference, it is a requirement.

</details>